# Finalised Model

In this notebook, we will train the finalised model with the optimal hyperparameters from the previous notebook. 
We will also attempt to explain how the model works with explainable AI using `SHAP (SHapley Additive exPlanations)`

## Define Final Pipeline

In [ ]:
FEATURES = ['int_rate','fico_range_high','inq_last_6mths','open_il_12m','acc_open_past_24mths','mort_acc','num_tl_op_past_12m','percent_bc_gt_75',
    'term','sub_grade']

NUMERICAL_FEATURES = ['int_rate','fico_range_high','inq_last_6mths','open_il_12m','acc_open_past_24mths','mort_acc','num_tl_op_past_12m','percent_bc_gt_75']

CATEGORICAL_FEATURES = ['term', 'sub_grade']



In [ ]:
impute = ColumnTransformer(
    [
        ("numerical_imputation", SimpleImputer(strategy='median'), NUMERICAL_FEATURES), 
        ("categorical_impuutation", SimpleImputer(strategy='most_frequent'), CATEGORICAL_FEATURES)
    ], 
    remainder='passthrough',  
    verbose_feature_names_out=False
)

encode = ColumnTransformer(
    [
        ("one_hot_encoding", OneHotEncoder(drop='first', sparse_output=False, dtype=np.int64), ['term']), 
        ("ordinal_encoding", OrdinalEncoder(dtype=np.int64), ['sub_grade']), 
    ],
    remainder='passthrough', 
    verbose_feature_names_out=False
)

scale = make_column_transformer(
    (RobustScaler(), NUMERICAL_FEATURES),
    remainder='passthrough', 
    verbose_feature_names_out=False
)

## Train Final Model on Full Dataset 

## Model Explainability (SHAP)

In [ ]:
model = XGBClassifier(**best_params)

y_train_numeric = y_train['loan_status'].map({'Charged Off':1, 'Fully Paid':0})

X_train_subset = X_train[FEATURES]

final_pipeline = Pipeline([
        ("imputation",impute),
        ("encoding",encode),
        ("feature_scaling", scale),
        ("xgboost_model", model)
    ])

final_pipeline.fit(X_train_subset, y_train_numeric)

In [ ]:
filepath_xtest = "/Users/thananpornsethjinda/Desktop/credit-risk-modeling/data/interim/xtest.csv" # filepath to train split 

filepath_ytest = "/Users/thananpornsethjinda/Desktop/credit-risk-modeling/data/interim/ytest.csv" # filepath to train split 

X_test = pd.read_csv(filepath_xtest)

y_test = pd.read_csv(filepath_ytest)

In [ ]:
X_test_subset = X_test[FEATURES]

y_test_numeric = y_test['loan_status'].map({'Charged Off':1, 'Fully Paid':0})

In [ ]:
y_train_pred = final_pipeline.predict(X_train_subset)
y_test_pred = final_pipeline.predict(X_test_subset)

In [ ]:
recall_train = recall_score(y_train_numeric, y_train_pred)
print(f"Recall (Train): {recall_train}")

recall_val = recall_score(y_test_numeric, y_test_pred)
print(f"Recall (Validation): {recall_val}")

precision_train = precision_score(y_train_numeric, y_train_pred)
print(f"Precision (Train): {precision_train}")

precision_val = precision_score(y_test_numeric, y_test_pred)
print(f"Precision (Validation): {precision_val}")

fbeta_train = fbeta_score(y_train_numeric, y_train_pred, beta=2)
print(f"F-Beta (Train): {fbeta_train}")

fbeta_val = fbeta_score(y_test_numeric, y_test_pred, beta=2)
print(f"F-Beta (Validation): {fbeta_val}")

matrix = confusion_matrix(y_test_numeric, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=matrix)
disp.plot()

In [ ]:
importances = plot_feature_importance(final_pipeline, model_type=lgb)
mlflow.log_figure(figure=importances, artifact_file="feature_importances.png")

# shap 

artifact_path = "model"

mlflow.sklearn.log_model(
    sk_model=final_pipeline,
    artifact_path="model_pipeline",
    input_example=X_train.iloc[:5] # Useful for UI preview
)

model_uri = mlflow.get_artifact_uri(artifact_path)